# OMNI Single-Year GIF Parsing

This notebook has been condensed to one task: load a single OMNI GIF file, extract its frames with PIL, convert the pixels to NumPy arrays, and organize the result into a clean pandas table.

## 1. Inspect OMNI GIF Data Files
## 2. Load a Single Year GIF with PIL

This condensed notebook starts with one local OMNI GIF file, verifies that PIL can open it, and then parses its frames into simple numeric summaries.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence

In [ ]:
gif_path = Path(r'c:\Users\sendt\Desktop\ML\SUN\omni_min__temp_mdlI4gZvxf.gif')
if not gif_path.exists():
    raise FileNotFoundError(f'Missing OMNI GIF: {gif_path}')

with Image.open(gif_path) as gif:
    print(f'File: {gif_path.name}')
    print(f'Format: {gif.format}')
    print(f'Mode: {gif.mode}')
    print(f'Size: {gif.size}')
    print(f'Frame count: {getattr(gif, "n_frames", 1)}')
    print(f'Animated: {getattr(gif, "is_animated", False)}')

## 3. Extract Frames and Basic Image Metadata

## 4. Parse Pixel Data into NumPy Arrays

The GIF is animated frame-by-frame. Each frame is extracted with PIL, summarized with basic metadata, and converted into a NumPy array so the pixel values can be handled numerically.

In [ ]:
frame_rows = []
frame_arrays = []
frame_meta = []

with Image.open(gif_path) as gif:
    for frame_index, frame in enumerate(ImageSequence.Iterator(gif)):
        rgb_frame = frame.convert('RGB')
        array = np.array(rgb_frame)
        frame_arrays.append(array)
        
        flattened = array.reshape(-1, 3)
        mean_rgb = flattened.mean(axis=0)
        std_rgb = flattened.std(axis=0)
        grayscale = array[..., 0] * 0.299 + array[..., 1] * 0.587 + array[..., 2] * 0.114
        
        frame_meta.append(
            {
                'frame_index': frame_index,
                'size': rgb_frame.size,
                'mode': rgb_frame.mode,
                'duration_ms': frame.info.get('duration', gif.info.get('duration')),
            }
        )
        frame_rows.append(
            {
                'frame_index': frame_index,
                'width': rgb_frame.size[0],
                'height': rgb_frame.size[1],
                'mode': rgb_frame.mode,
                'duration_ms': frame.info.get('duration', gif.info.get('duration')),
                'mean_r': float(mean_rgb[0]),
                'mean_g': float(mean_rgb[1]),
                'mean_b': float(mean_rgb[2]),
                'std_r': float(std_rgb[0]),
                'std_g': float(std_rgb[1]),
                'std_b': float(std_rgb[2]),
                'grayscale_mean': float(grayscale.mean()),
                'nonwhite_ratio': float(np.mean(np.any(array < 250, axis=2))),
            }
        )

print(f'Extracted {len(frame_rows)} frames from {gif_path.name}')

## 5. Map Parsed Values to a Time or Year Index

Because a GIF is an image stream, each parsed frame is mapped onto a position inside the selected year so the output stays ordered and easy to inspect.

## 6. Build a Clean DataFrame for One Year

The parsed frame summaries are organized into a pandas DataFrame with a frame index, an approximate time index, and simple pixel-derived numeric columns.

In [ ]:
year_start = pd.Timestamp('2020-01-01', tz='UTC')
year_end = pd.Timestamp('2020-12-31 23:59:59', tz='UTC')

df = pd.DataFrame(frame_rows)
if not df.empty:
    df['frame_index'] = np.arange(len(df), dtype=int)
    if len(df) == 1:
        df['time_index'] = [year_start]
    else:
        df['time_index'] = pd.date_range(year_start, year_end, periods=len(df), tz='UTC')
    df['year_fraction'] = df['frame_index'] / max(len(df) - 1, 1)
    df['rgb_mean_scalar'] = df[['mean_r', 'mean_g', 'mean_b']].mean(axis=1)
    df['rgb_spread'] = df[['std_r', 'std_g', 'std_b']].mean(axis=1)

print(df.head())

## 7. Validate the Parsed Output

This final step checks the parsed frame table for row count, missing values, and a small sample so we can confirm the one-year OMNI GIF was loaded and converted into a numeric structure.

In [ ]:
if len(df) == 0:
    raise ValueError('No frames were parsed from the GIF.')

print(f'Parsed rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print('Missing values by column:')
print(df.isna().sum())
print('\nFirst rows:')
print(df.head())